# Annotate TFs
## AnimalTFDB v4.0 (human and mouse)
### Author: Martin Loza
### Date: 25/12/09

Once we get the list of PCG near to a ncRNA, we will annotate the ones related to TFs

In [30]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
})

# Local variables 
seed = 777
date = "251209"

red = "#E41A1C"
blue = "#377EB8"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

in_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/distances/single_pcg_gene_transcripts/"
tf_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/AnimalTFDB/"
out_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/"

# Local Functions


### Load and setup the data

In [40]:
# Load the gene annotations with distances. 
# In this df, we related ncRNA transcripts to the closest PCG transcripts.
# We have filtered to keep only one transcript per PCG gene (the closest one).

# We have different species, so let's create a list to store the data
data_list = list()

# Search for the available files
files <- list.files(in_dir)

# Load the data for each species
for (file in files) {
    # Remove the underscore and everything after it to get the species names
    species_name <- str_replace(file, "_.*", "")
    data_list[[species_name]] <- read.table(file.path(in_dir, file), sep = "\t", header = TRUE, 
                                            stringsAsFactors = FALSE, quote = "", 
                                            comment.char = "", fill = TRUE, row.names = NULL)
}

head(data_list[["dog"]], 3)


,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>
1,1,ENSCAFT00845000004,41931353,,1,lncRNA,ENSCAFT00845037491,AKAP12,41882972,-48381,-48381
2,1,ENSCAFT00845000004,41931353,,1,lncRNA,ENSCAFT00845000019,ARMT1,41984839,53486,53486
3,1,ENSCAFT00845000004,41931353,,1,lncRNA,ENSCAFT00845000026,CCDC170,42020714,89361,89361


For now, let's just keep human and mouse

In [41]:
# Select species of interest
selected_species <- c("human", "mouse")
data_list <- data_list[selected_species]

In [43]:
unique(data_list[["mouse"]]$chromosome)

[1] "6"          "2"          "10"         "11"         "12"        
 [6] "13"         "14"         "15"         "18"         "1"         
[11] "3"          "5"          "7"          "X"          "8"         
[16] "9"          "16"         "17"         "4"          "19"        
[21] "MT"         "Y"          "GL456210.1" "GL456221.1" "JH584304.1"
[26] "GL456211.1"

In [47]:
human_chromosomes <- c(1:22, "X", "Y")
mouse_chromosomes <- c(1:19, "X", "Y")

# Select data from selected chromosomes only
data_list[["human"]] <- data_list[["human"]] %>%
    filter(chromosome %in% as.character(human_chromosomes)) 
data_list[["mouse"]] <- data_list[["mouse"]] %>%
    filter(chromosome %in% as.character(mouse_chromosomes)) 

cat("Unique human chromosomes in data:", unique(data_list[["human"]]$chromosome), "\n")
cat("Unique mouse chromosomes in data:", unique(data_list[["mouse"]]$chromosome), "\n")

Unique human chromosomes in data: 19 12 14 1 7 20 2 11 22 Y 16 17 X 3 6 10 21 9 5 8 15 4 18 13 
Unique mouse chromosomes in data: 6 2 10 11 12 13 14 15 18 1 3 5 7 X 8 9 16 17 4 19 Y 


Now, let's load the TF annotations from AnimalTFDB4

In [48]:
# Search for the available files
files <- list.files(tf_dir)
# We have renamed the files to contain "human" and "mouse" in their names

# Create a list to store TF data
tf_data_list <- list()

for(specie in selected_species) {
    # Find the corresponding TF file
    tf_file <- files[grepl(specie, files)]
    # Load the TF data
    tf_data <- read.table(file.path(tf_dir, tf_file), sep = "\t", header = TRUE, 
                          stringsAsFactors = FALSE, quote = "", 
                          comment.char = "", fill = TRUE, row.names = NULL)
    
    # Select columns of interests
    sel_cols <- c("Symbol", "Ensembl", "Family")
    tf_data <- tf_data %>%
        select(all_of(sel_cols))
    
    # Add the data to the list
    tf_data_list[[specie]] <- tf_data
}

head(tf_data_list[["human"]], 3)

,Symbol,Ensembl,Family
,<chr>,<chr>,<chr>
1,ATF1,ENSG00000123268,TF_bZIP
2,,ENSG00000254553,ZBTB
3,SOX3,ENSG00000134595,HMG


### Annotate TFs

We want a simple annotation. 

We want to annotate whether a PCG is or not a TFs. If it's a TF, we annotate the family for potential downstream analyses.

In [49]:
head(data_list[["human"]], 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913


In [50]:
# let's investigate matching between gene annotations
cat("Number of humant TFs:", nrow(tf_data_list[["human"]]), "\n")
cat("Number of human TFs related to PCG transcripts:", sum(tf_data_list[["human"]]$Symbol %in% data_list[["human"]]$pcg_gene_name), "\n")
cat("Number of mouse TFs:", nrow(tf_data_list[["mouse"]]), "\n")
cat("Number of mouse TFs related to PCG transcripts:", sum(tf_data_list[["mouse"]]$Symbol %in% data_list[["mouse"]]$pcg_gene_name), "\n")

Number of humant TFs: 1659 
Number of human TFs related to PCG transcripts: 1650 
Number of mouse TFs: 1611 
Number of mouse TFs related to PCG transcripts: 1563 


Ok, these numbers are promising!

In [51]:
# Let's create a new list to store the annotated data
annotated_data_list <- list()

# Annotate the distance data with TF information
for(specie in selected_species) {
    # Get the distance data
    dist_data <- data_list[[specie]]
    # Get the TF data
    tf_data <- tf_data_list[[specie]]

    # Setup for joining
    transfer_cols <- c("Symbol", "Family")
    tf_data <- tf_data %>%
        select(all_of(transfer_cols)) %>%
        dplyr::rename(pcg_gene_name = Symbol)
    
    # Annotate the distance data
    annotated_data <- dist_data %>%
        left_join(tf_data, by = "pcg_gene_name")

    # Create simple annotation column
    annotated_data <- annotated_data %>%
        mutate(is_TF = ifelse(is.na(Family), FALSE, TRUE))
    
    # Store the annotated data
    annotated_data_list[[specie]] <- annotated_data
}

Warning message in left_join(., tf_data, by = "pcg_gene_name"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 75 of `x` matches multiple rows in `y`.
ℹ Row 2 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


In [52]:
tt_human <- annotated_data_list[["human"]] %>% filter(!is.na(Family))
tt_mouse <- annotated_data_list[["mouse"]] %>% filter(!is.na(Family))

cat("Number of unique human TFs in annotated data:", length(unique(tt_human$pcg_gene_name)), "\n")
cat("Number of unique mouse TFs in annotated data:", length(unique(tt_mouse$pcg_gene_name)), "\n")
cat("Number of human PCG transcripts annotated as TFs:", nrow(tt_human), "\n")
cat("Number of mouse PCG transcripts annotated as TFs:", nrow(tt_mouse), "\n")

Number of unique human TFs in annotated data: 1629 
Number of unique mouse TFs in annotated data: 1563 
Number of human PCG transcripts annotated as TFs: 1310526 
Number of mouse PCG transcripts annotated as TFs: 261005 


We have several transcripts related to same PCGs. There can be also multiple ncRNA, since the TSS sometimes are very near to each other

Let's save the annotated distances

In [53]:
# Save the annotated data
for(specie in selected_species) {
    out_file <- file.path(out_dir, paste0(specie, "_ncRNA_pcg_distances_annotated_TFs_", date, ".tsv"))
    write.table(annotated_data_list[[specie]], file = out_file, sep = "\t", 
                quote = FALSE, row.names = FALSE, col.names = TRUE)
}